# Is a high-order cold-mask OPD degenerate with `primary_opd`?

At the best fit of `calibrator-fit-bootstrap.ipynb`, add a second Fourier-basis OPD layer **downstream of the
occulter** (`cold_mask_fourier`, `n_new` x `n_new` modes, after `cold_mask_opd`) and ask, with the Gauss-Newton
Hessian of the joint `[primary_opd, cold_mask_fourier]` block, how many independent dimensions it adds and
whether fitting it would reduce the residuals.

In [ ]:
import sys
sys.path.insert(0, '..')

In [ ]:
# Basic imports
import jax.numpy as np
import jax.random as jr
import jax.scipy as jsp
import jax
import numpy

jax.config.update("jax_enable_x64", True)


# Optimisation imports
import zodiax as zdx
import optax
import optimistix as optx

# dLux imports
import dLux as dl
import dLux.utils as dlu

# Visualisation imports
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib
import chainconsumer as cc


%matplotlib inline
plt.rcParams['image.cmap'] = 'inferno'
plt.rcParams["font.family"] = "serif"
plt.rcParams["image.origin"] = 'lower'
plt.rcParams['figure.dpi'] = 72
plt.rcParams["font.size"] = 24

from detectors import *
from apertures import *
from models import *
from stats import posterior
from fitting import *
from plotting import *
from plotting import _panel, _signed_panel   # not exported by *
from spectra import *

import interpax as ipx

## Settings and data

In [ ]:
wf_wid = 512
wid = 80
oversample = 4

nwavels = 3
npoly=1

n_modes = 45
n_zernikes = 30

n_new = 45        # Fourier modes per axis of the new cold-mask OPD block
batch_size = 2    # Jacobian tangents evaluated per batch (memory vs speed)

resolved_wid = 1

detector = NICMOSDetector(oversample, wid)

spectrum_basis = np.ones((nwavels, npoly))

ddir = '../data/NICMOS-LAPL-DD2/LAPL_DATA_DD2/comtemp_flats-DD2/'
flatdir = '../data/NICMOS-LAPL-DD2/LAPL_HOLEFLATS_DD2/'

exposures_single = [
    exposure_from_file(ddir + 'n8jv53xsq_o_clc_calf.fits', SinglePointFit(spectrum_basis, "F187N"), crop=wid, extra_bad=np.load("bad_map.npy"), flatcorr=flatdir),
]

## Model with a cold-mask Fourier OPD

A copy of `NICMOSCoronagraph` with one extra layer, `cold_mask_fourier`, placed after `cold_mask_opd` (downstream
of the occulter, before `cold_mask_tilt`). `propagate` splits the layers at `"occulter"` and treats the last layer
as the final MFT, so the inherited `propagate` applies the new layer in the post-occulter stage unchanged.

In [ ]:
class ColdMaskFourierCoronagraph(NICMOSCoronagraph):
    def __init__(self, wf_npixels, psf_npixels, oversample, n_modes=12, n_zernikes=1., n_new=45, wl_batch=None, remat=False):
        diameter = 3.
        # Layer list copied from apertures.NICMOSCoronagraph (no turboklip), plus "cold_mask_fourier"
        layers = [
            ("primary", HSTMainAperture(transformation=dl.CoordTransform(rotation=np.pi/4), softening=5)),
            ("primary_tilt", dl.Tilt(angles=(0.,0.))),
            ("primary_opd", dl.FourierBasis(wf_npixels, n_modes=n_modes)),
            ("primary_low", ScanAberratedAperture(
                dl.layers.CircularAperture(1.2, transformation=dl.CoordTransform(translation=np.zeros(2))),
                noll_inds=np.arange(4, 4+n_zernikes),
                coefficients=np.zeros(n_zernikes),
            )),
            ("flip", dl.Flip(axes=(0,1))),
            ("occulter", SoummerFastObstruction([
                ("prop1", dl.MFT(256, focal_length=24*2.4, pixel_scale=1.5e-6)),
                ("occulter", CLIMBOcculter(dlu.arcsec2rad(0.3)*24*2.4, np.zeros(1), np.zeros(1))),
                ("prop1", dl.MFT(wf_npixels, focal_length=24*2.4, pixel_scale=diameter/wf_npixels)),
            ])),
            ("cold_mask", NIC2ColdMask(transformation=dl.CoordTransform(translation=np.asarray((-0.05,-0.05)), rotation=np.pi/4, compression=np.asarray([1.,1.])), softening=5, normalise=False)),
            ("cold_mask_opd", ScanAberratedAperture(
                dl.layers.CircularAperture(1.2, transformation=dl.CoordTransform(translation=np.asarray((-0.05, -0.05)))),
                noll_inds=np.arange(4, 20),
                coefficients=np.zeros(1),
            )),

            # NEW: high-order Fourier OPD on the cold mask, after the occulter
            ("cold_mask_fourier", dl.FourierBasis(wf_npixels, n_modes=n_new)),

            ("cold_mask_tilt", dl.Tilt(angles=(0.,0.))),
            ("prop1", dl.MFT(psf_npixels*oversample, focal_length=45.7*2.4, pixel_scale=40e-6/oversample)),
        ]
        # Skip NICMOSCoronagraph.__init__ (it would build the old layer list)
        dl.LayeredOpticalSystem.__init__(self, wf_npixels, diameter, layers)
        self.wl_batch = wl_batch
        self.remat = bool(remat)


optics = ColdMaskFourierCoronagraph(wf_wid, wid, oversample, n_modes=n_modes, n_zernikes=n_zernikes, n_new=n_new)

# Checks: zero coefficients reproduce NICMOSCoronagraph bit-for-bit, and the new layer changes the image
wavelengths = np.linspace(1.85e-6, 1.89e-6, nwavels)
# (cold_mask_opd is built with 1 coefficient but 16 Noll indices; the fit sets all 16, so do the same here)
old = NICMOSCoronagraph(wf_wid, wid, oversample, n_modes=n_modes, n_zernikes=n_zernikes)
old, new = [o.set("cold_mask_opd.coefficients", np.zeros(16)) for o in (old, optics)]
psf_old, psf_new = old.propagate(wavelengths), new.propagate(wavelengths)
print("zero coefficients: bit-for-bit equal:", bool(np.all(psf_old == psf_new)))

coeffs = 5e-9 * jr.normal(jr.PRNGKey(0), (n_new, n_new))
psf_poked = new.set("cold_mask_fourier.coefficients", coeffs).propagate(wavelengths)
print("5 nm random coefficients: max |change| / max image =", float(np.abs(psf_poked - psf_new).max() / psf_new.max()))

## Fit class

`cold_mask_fourier` is a global parameter in nm (x1e-9), like `primary_opd`, so both blocks share one metric in the
joint Hessian. The piston is zeroed in `update_optics` (it is a constant phase, which does nothing).

In [ ]:
class ColdMaskFourierFit(SinglePointFit):
    PARAM_KEYS = SinglePointFit.PARAM_KEYS | {"cold_mask_fourier": "global"}

    def update_optics(self, model, exposure):
        optics = super().update_optics(model, exposure)
        if "cold_mask_fourier" in model.params.keys():
            coefficients = model.get(self.map_param(exposure, "cold_mask_fourier"))*1e-9
            coefficients = coefficients.at[0,0].set(0.)
            optics = optics.set("cold_mask_fourier.coefficients", coefficients)
        return optics


exposures = [exp.set("fit", ColdMaskFourierFit(spectrum_basis, "F187N")) for exp in exposures_single]

## Best fit

Parameters from the `calibrator-fit-bootstrap.ipynb` initial dict, overwritten by `params.npy` (its final
`params_history[-1]`; `outer_radius`, `primary_pad`, `primary_outer` are not optimised there so stay at their initial values),
plus `cold_mask_fourier = 0`.

In [ ]:
params = {
    "spectrum": {},
    "primary_opd": {},
    "primary_low": {},
    "primary_tilt": {},
    "cold_mask_opd": {},
    "cold_mask_tilt": {},
    "cold_mask_shift": {},
    "cold_mask_rot": {},
    "cold_mask_shear": {},
    "cold_mask_scale": {},
    "primary_rot": {},
    "primary_shear": {},
    "cold_mask_fourier": {},

    "bias": {},
    "occulter_radius": 0.8,
    "occulter_coeffs": np.zeros(4)+1e-3,
    "fnumber": 45.7,
    "anisotropy": 1.+1e-5,

    "outer_radius": 1.2*0.9768,
    "secondary_radius": 0.365*1.2,
    "spider_width": 0.072*1.2,
    "primary_spider": 0.0256,
    "primary_secondary": 0.330*1.2,
    "primary_outer": 1.2,
    "primary_pad": 0.065*1.2,
}

for idx, exp in enumerate(exposures):
    params["spectrum"][exp.fit.get_key(exp, "spectrum")] = (np.zeros(npoly)).at[0].set((np.nansum(exp.data)/nwavels)*210)
    params["primary_tilt"][exp.fit.get_key(exp, "primary_tilt")] = np.array([-0.05, -0.75])*0.075
    params["cold_mask_tilt"][exp.fit.get_key(exp, "cold_mask_tilt")] = np.array([
        np.array(exp.hdr["TARSIAFY"],dtype=float) - (256-44),
        np.array(exp.hdr["TARSIAFX"],dtype=float) - (256-181),
    ])*0.075

    params["primary_opd"][exp.fit.get_key(exp, "primary_opd")] = np.zeros((n_modes, n_modes))
    params["primary_low"][exp.fit.get_key(exp, "primary_low")] = np.zeros((n_zernikes))
    params["cold_mask_opd"][exp.fit.get_key(exp, "cold_mask_opd")] = np.zeros(16).at[0].set(120.)

    params["cold_mask_shift"][exp.fit.get_key(exp, "cold_mask_shift")] = np.array([13.,10.])
    params["cold_mask_rot"][exp.fit.get_key(exp, "cold_mask_rot")] = 3.2
    params["primary_rot"][exp.fit.get_key(exp, "primary_rot")] = -0.6
    params["cold_mask_scale"][exp.fit.get_key(exp, "cold_mask_scale")] = np.asarray([1.,1.])
    params["cold_mask_shear"][exp.fit.get_key(exp, "cold_mask_shear")] = np.asarray([0.06,-0.06])
    params["primary_shear"][exp.fit.get_key(exp, "primary_shear")] = np.asarray([0.,0.])
    params["bias"][exp.fit.get_key(exp, "bias")] = 0.

    # the new block starts at zero, so the image is exactly the bootstrap best fit
    params["cold_mask_fourier"][exp.fit.get_key(exp, "cold_mask_fourier")] = np.zeros((n_new, n_new))

# Final stage of the bootstrap fit (params_history[-1], saved as params.npy)
params = params | np.load("params.npy", allow_pickle=True).item()

model = set_array(NICMOSModel(exposures, params, optics, detector))
params = ModelParams(params)
exp = exposures[0]

In [ ]:
# Sanity check: residual sigma should match the bootstrap fit (~4.85)
plot_comparison_detailed(model, params, exposures, percentile=100, quadrature=False, wf_size=wf_wid)

## Jacobian

Every parameter that the bootstrap's final stage optimises, plus the new block. `model_jacobian` is forward mode
(reverse-mode gradients are wrong for `primary_rot`/`primary_shear`/`primary_spider` on this JAX build); its
`opd_sample` applies to `primary_opd` only, so we pass all `n_modes` x `n_modes` indices, and the 2-D
`cold_mask_fourier` is enumerated in full automatically.

In [ ]:
fit_keys = [
    "primary_opd", "cold_mask_fourier",     # the two blocks under test
    "spectrum", "primary_tilt", "cold_mask_tilt", "cold_mask_opd", "bias", "cold_mask_shift", "cold_mask_rot",
    "primary_rot", "primary_low", "cold_mask_scale", "cold_mask_shear", "primary_shear", "occulter_radius",
    "occulter_coeffs", "secondary_radius", "spider_width", "primary_spider", "primary_secondary", "fnumber", "anisotropy",
]
sub = {k: params[k] for k in fit_keys}
all_opd = [(i, j) for i in range(n_modes) for j in range(n_modes)]

columns, jacobian = model_jacobian(model, ModelParams(sub), exp, opd_sample=all_opd, batch_size=batch_size)
print(len(columns), "columns", jacobian.shape)

## Fisher information

Whiten by the pixel errors and keep good pixels only (`model_jacobian` returns NaN at bad pixels). Rows of `Jw`
are pixels, columns are parameters, so `F = Jw^T Jw` is the Gauss-Newton Hessian of `0.5 chi^2`. We use Gauss-Newton
rather than the full Hessian because it is positive semi-definite and the dropped residual term is not small at an
unconverged fit (it would add indefinite directions that obscure the degeneracy structure). Both pistons (constant
phase) and any exactly-zero columns are dropped. Blocks: `o` = `primary_opd`, `n` = `cold_mask_fourier`,
`e` = every other parameter.

In [ ]:
good = ~numpy.asarray(exp.bad)
err = numpy.asarray(exp.err)[good]
Jw = (jacobian[:, good] / err).T.astype(numpy.float64)       # (n_good_pixels, n_columns)
N = Jw.shape[0]

names = numpy.array([name for name, _ in columns])
index = [idx for _, idx in columns]
alive = (numpy.linalg.norm(Jw, axis=0) > 0) & ~numpy.array([(n in ("primary_opd", "cold_mask_fourier")) and i == (0, 0) for n, i in columns])

o = numpy.where(alive & (names == "primary_opd"))[0]
n = numpy.where(alive & (names == "cold_mask_fourier"))[0]
e = numpy.where(alive & ~numpy.isin(names, ["primary_opd", "cold_mask_fourier"]))[0]
m = numpy.concatenate([o, e])                                  # everything except the new block
print(f"{N} good pixels; columns: o {len(o)}, n {len(n)}, e {len(e)}")

F = Jw.T @ Jw
block = lambda a, b: F[numpy.ix_(a, b)]

# Whitened residual of the current best fit
r = (numpy.asarray(exp.data - exp.fit(model, exp))[good]) / err
print(f"chi2/N of the current fit: {r @ r / N:.3f}")

## Analysis

### Eigenvalue spectra
An eigenvalue is the information on that mode combination in units of 1/(1 nm)^2: eigenvalue 1 means the
mode is constrained to 1 nm at 1 sigma, 1e-2 means 10 nm.

In [ ]:
RCOND = 1e-12

def eig_desc(M):
    w, V = numpy.linalg.eigh(M)
    return w[::-1], V[:, ::-1]

def pinv_psd(M, rcond=RCOND):
    w, V = numpy.linalg.eigh(M)
    keep = w > rcond * w.max()
    return (V[:, keep] / w[keep]) @ V[:, keep].T

w_o, _ = eig_desc(block(o, o))
w_n, _ = eig_desc(block(n, n))
on = numpy.concatenate([o, n])
w_j, V_j = eig_desc(block(on, on))

fig, ax = plt.subplots(figsize=(14, 10), layout='compressed')
for w, label in [(w_o, r"$F_{oo}$ (primary_opd)"), (w_n, r"$F_{nn}$ (cold_mask_fourier)"), (w_j, "joint [o, n]")]:
    ax.plot(numpy.log10(numpy.maximum(w, 1e-30)), label=label)
ax.axhline(0, c='k', ls='--', label=r"1 nm-equivalent at 1$\sigma$")
ax.axhline(-2, c='gray', ls=':', label="10 nm")
ax.set_ylim(-12, None)
ax.set_xlabel("eigenvalue index")
ax.set_ylabel(r"$\log_{10}$ eigenvalue (nm$^{-2}$)")
ax.legend(fontsize=16);

### Degeneracy
Schur complement `S = F_nn - F_no F_oo^-1 F_on` is the information on the new block that `primary_opd` cannot
absorb; `S_all` marginalises `primary_opd` and all the other parameters (`m = o + e`). The retained-information
ratio is the generalised eigenvalue of `(S_all, F_nn)`, i.e. one minus the squared canonical correlation between
the column spaces of `Jw_n` and `Jw_[o,e]`: about 0 means the new mode is degenerate, about 1 means independent.

In [ ]:
schur = lambda keep: block(n, n) - block(n, keep) @ pinv_psd(block(keep, keep)) @ block(keep, n)
S = schur(o)
S_all = schur(m)

w_S, _ = eig_desc(S)
w_Sall, V_Sall = eig_desc(S_all)

fig, ax = plt.subplots(figsize=(14, 10), layout='compressed')
for w, label in [(w_n, r"$F_{nn}$"), (w_S, r"$S$ (o marginalised)"), (w_Sall, r"$S_{all}$ (o + e marginalised)")]:
    ax.plot(numpy.log10(numpy.maximum(w, 1e-30)), label=label)
ax.axhline(0, c='k', ls='--')
ax.axhline(-2, c='gray', ls=':')
ax.set_ylim(-12, None)
ax.set_xlabel("eigenvalue index")
ax.set_ylabel(r"$\log_{10}$ eigenvalue (nm$^{-2}$)")
ax.legend(fontsize=16);

# Retained-information ratio via orthonormal bases of the two column spaces (singular values above sqrt(RCOND) of the largest)
def col_basis(J):
    U, s, _ = numpy.linalg.svd(J, full_matrices=False)
    return U[:, s > numpy.sqrt(RCOND) * s[0]]

Q_m, Q_n = col_basis(Jw[:, m]), col_basis(Jw[:, n])
rho = numpy.linalg.svd(Q_m.T @ Q_n, compute_uv=False)               # canonical correlations
ratio = numpy.sort(1 - rho**2)[::-1]

fig, axs = plt.subplots(1, 2, figsize=(20, 9), layout='compressed')
axs[0].plot(ratio); axs[0].set_xlabel("sorted new-block direction"); axs[0].set_ylabel(r"retained information $1-\rho^2$")
axs[1].hist(ratio, bins=40); axs[1].set_xlabel(r"retained information $1-\rho^2$"); axs[1].set_ylabel("directions");
print(f"directions in the new block's column space: {len(ratio)} of {len(n)}; median retained {numpy.median(ratio):.3f}; "
      f">0.5: {(ratio > .5).sum()}, <0.01: {(ratio < .01).sum()}")

### Dimensionality

In [ ]:
count = lambda w, t: int((w > t).sum())
print(f"{'eigenvalues >':<16}{'1':>8}{'1e-2':>8}")
for label, w in [("F_oo", w_o), ("F_nn", w_n), ("joint [o, n]", w_j), ("S_all", w_Sall)]:
    print(f"{label:<16}{count(w, 1):>8}{count(w, 1e-2):>8}")
print(f"{'added dims':<16}{count(w_j, 1) - count(w_o, 1):>8}{count(w_j, 1e-2) - count(w_o, 1e-2):>8}   (joint - F_oo)")

### Would it help?
Linearised fit of the current whitened residual `r`: one damped Gauss-Newton step,
`delta = (F + lam diag(F))^-1 J^T r`, then `chi2 = |r - J delta|^2`, with `[o, e]` only versus `[o, e, n]`
at the same relative damping `lam`. A pure-noise fit of `k` independent dimensions would lower chi2 by about `k`.

In [ ]:
def gn_step_gain(cols, lam):
    J = Jw[:, cols]
    Fc = J.T @ J
    delta = numpy.linalg.solve(Fc + lam * numpy.diag(numpy.diag(Fc)), J.T @ r)
    return (r @ r - numpy.sum((r - J @ delta)**2)) / N

print(f"chi2/N before: {r @ r / N:.4f}")
print(f"{'lam':>8}{'[o,e]':>12}{'[o,e,n]':>12}{'extra':>12}   (decrease in chi2/N)")
for lam in [1e-6, 1e-4, 1e-2]:
    a, b = gn_step_gain(m, lam), gn_step_gain(numpy.concatenate([m, n]), lam)
    print(f"{lam:>8.0e}{a:>12.4f}{b:>12.4f}{b - a:>12.4f}")
print(f"noise expectation of the extra gain: ~{count(w_Sall, 1) / N:.4f} (S_all eigenvalues > 1) to {len(ratio) / N:.4f} (all new directions)")

## Eigenmodes
Joint `[o, n]` eigenvectors spanning the spectrum. Columns: the `primary_opd` part (nm, over the primary), the
`cold_mask_fourier` part (nm, over the cold mask) and the image signature `Jw v` (z-score per unit `|v|`).

In [ ]:
coords = dlu.pixel_coords(wf_wid, model.optics.diameter)
final_optics = exp.fit.update_optics(model, exp)
primary_support = final_optics.primary.transmission(coords, 2.4/wf_wid) > .5
cold_mask_support = final_optics.cold_mask.transmission(coords, 2.4/wf_wid) > .5

def coeff_map(v, cols, size):
    """Scatter a vector over `cols` into a (size, size) coefficient array (dropped piston = 0)."""
    c = numpy.zeros((size, size))
    for val, k in zip(v, cols):
        c[index[k]] = val
    return c

def pupil(layer, c, support):
    opd = layer.set("coefficients", np.asarray(c)).eval_basis()
    return numpy.where(support, numpy.asarray(opd), numpy.nan)

def to_image(vec):
    im = numpy.full(good.shape, numpy.nan)
    im[good] = vec
    return im

n_valid = int((w_j > RCOND * w_j[0]).sum())
k1 = count(w_j, 1)
picks = numpy.unique(numpy.clip([0, 1, 3, k1 - 5, k1, k1 + 5, n_valid - 20, n_valid - 1], 0, n_valid - 1))

fig, axs = plt.subplots(len(picks), 3, figsize=(30, 10 * len(picks)), layout='compressed', squeeze=False)
for row, k in zip(axs, picks):
    v = V_j[:, k]
    frac = numpy.sum(v[len(o):]**2)
    tag = f"mode {k}, eig {w_j[k]:.1e}, |v_n|$^2$={frac:.2f}"
    _signed_panel(row[0], pupil(final_optics.primary_opd, coeff_map(v[:len(o)], o, n_modes), primary_support), tag, label="primary_opd OPD (nm)")
    _signed_panel(row[1], pupil(final_optics.cold_mask_fourier, coeff_map(v[len(o):], n, n_new), cold_mask_support), "", label="cold_mask_fourier OPD (nm)")
    _signed_panel(row[2], to_image(Jw[:, on] @ v), "image signature", label="z-score per unit |v|")

In [ ]:
# The most independently constrained new-block modes: top S_all eigenvectors. The signature is the part of
# Jw_n v that the other parameters cannot reproduce, whose squared norm is the S_all eigenvalue.
k_top = 6
fig, axs = plt.subplots(k_top, 2, figsize=(20, 10 * k_top), layout='compressed', squeeze=False)
for k, row in enumerate(axs):
    v = V_Sall[:, k]
    sig = Jw[:, n] @ v
    sig = sig - Q_m @ (Q_m.T @ sig)
    _signed_panel(row[0], pupil(final_optics.cold_mask_fourier, coeff_map(v, n, n_new), cold_mask_support), f"S_all mode {k}, eig {w_Sall[k]:.2e}", label="OPD (nm)")
    _signed_panel(row[1], to_image(sig), "independent image signature", label="z-score per unit |v|")

## Conclusion

Results at the bootstrap best fit (6201 good pixels, 4123 columns, chi2/N = 23.6, i.e. residual sigma ~4.85).

- **Dimensionality** (eigenvalues > 1 / > 1e-2): `F_oo` 948 / 1072, `F_nn` 863 / 1073, joint 1548 / 1970, `S_all` 571 / 912. The new block adds **600 dimensions at 1 nm and 898 at 10 nm**, so it is far from fully absorbed even though both blocks are 45 x 45 and the data have only ~6200 pixels.
- **Degeneracy**: strongly but not completely degenerate. Marginalising `primary_opd` alone removes about 1/3 of the 1-nm dimensions from `F_nn` (863 to ~600); marginalising everything else leaves 571. Per direction, the median retained information is only 0.047 (195 directions above 0.5, 420 below 0.01), and `F_nn` has only 1128 numerically non-null directions out of 2024. Low-order and mid-order joint modes are mixtures of the same pattern in both layers (`|v_n|^2` ~ 0.05 for the best constrained modes, ~0.8 near eigenvalue 1), which is the expected sign of near-degeneracy: a pupil phase and an equal-and-opposite cold-mask phase produce similar speckle. The independently constrained `S_all` modes sit at the edges of the cold-mask support (the right-hand limb, near the pads) and have localised image signatures, i.e. they probe where the cold mask differs geometrically from the primary.
- **Predicted improvement**: one damped GN step reduces chi2/N by 7.8 (`[o, e]`, lam 1e-4) versus 21.1 (`[o, e, n]`): an extra 13 (10 to 17 over lam 1e-6 to 1e-2). Pure noise would give only ~0.1 to 0.2 per N in units where chi2/N = 1, i.e. ~2 to 4 here after scaling by the current chi2/N of 23.6.
- **Caveat**: the gain is linear, in-sample, and with ~4000 free coefficients against 6200 pixels it mostly measures how much structured residual (and unmodelled noise) 600 more independent dimensions can soak up, not that the cold mask truly has high-order OPD.

Whether it is worth adding: the extension is not redundant, so it would lower chi2 substantially, but it is also not well determined (eigenvalues of the extra modes lie mostly between 1 and 1e-4 nm^-2, i.e. 1 to 100 nm uncertainty per mode). Add it only with regularisation (a smoothness or power-law prior) or at a much lower order, and check against held-out exposures, before trusting it as physical.